In [ ]:
from collections import deque
from pathlib import Path
import sys

import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

working_directory = Path.cwd().resolve()
PROJECT_ROOT = (
    working_directory
    if (working_directory / "src" / "ncpu_computer").is_dir()
    else working_directory.parent
)
SOURCE_ROOT = PROJECT_ROOT / "src"
if not (SOURCE_ROOT / "ncpu_computer").is_dir():
    raise RuntimeError("Run this notebook from ncpu-em-2d or ncpu-em-2d/notebooks")
sys.path.insert(0, str(SOURCE_ROOT))

import torch
from IPython.display import Image as DisplayImage, display

from ncpu_computer import (
    ExperimentConfig, GeometryConfig, ModelConfig, MultiTaskDataset,
    NeuralCellularAutomaton, TapeLayout, TestCase, Trainer, TrainingConfig,
    binary_tasks, encode_strings, evaluate_cases, format_results, infer,
    interpret_tape, load_model, save_gif, validate_experiment, validate_task_specs,
)

# Tasks and type-independent direct tape interface
TASKS = (
    ("copy", 1.0),
    ("bit_not", 1.0),
    ("reverse", 2.0),
    ("reverse_not", 2.0),
    ("shift_left_zero", 1.0),
    ("shift_right_zero", 1.0),
    ("gray_encode", 1.0),
)
TASK_NAMES, TASK_WEIGHTS = validate_task_specs(TASKS)
EXPERIMENT_NAME = "binary_multitask"
STRIDE = 2
VERTICAL_SPACE = 1
HORIZONTAL_SPACE = 2
PROGRAM_START = 1  # 0: repeat from x=0; 1: zero marker at x=0, then repeat
WRAP_Y = False

# Repeated task program and local NCA rule
PROGRAM_CHANNELS = 3
PROGRAM_MODE = "learned_read_only"  # zero, learned_read_only, learned_mutable
PROGRAM_INIT_STD = 0.3
COMPUTATION_CHANNELS = 4
HIDDEN_SIZE = 96
FIXED_KERNELS = ("identity", "sobel_x", "sobel_y")
FIXED_LAPLACIAN = False
LEARNABLE_KERNELS = 1
LEARNABLE_KERNEL_INIT = "laplacian"
ACTIVATION = "relu"  # relu or softplus
GATE = "none"
GATE_BIAS = 1.0
FIRE_RATE = 0.9
STATE_LEAK = 0.1
MAX_ABS_STATE = 10.0

# One optimizer update accumulates one independently varied batch per base pair
# Every active task must leave at least one blank after both input and output
UPDATES = 2000
BATCH_SIZE_PER_TASK = 24
BASE_TAPE_SLOTS = (5, 7, 8, 9, 11)
BASE_INPUT_MAX_LENGTHS = (3, 5, 6, 7, 8)
TAPE_VARIATION = 0.20
INPUT_VARIATION = 0.20
FREE_STEPS_PER_TAPE_SLOT = 6.0
TIME_VARIATION = 0.20
SUPERVISION_RATIO = 1.5
LR_POINTS = (
    (0.00, 2e-3),
    (0.2, 1e-3),
    (0.6, 3e-4),
    (0.65, 6e-4),
    (0.7, 1e-4),
    (0.8, 5e-5),
    (1.0, 1e-5),
)
LR_INTERPOLATION = "cosine"  # cosine or linear
WEIGHT_DECAY = 1e-5
PROGRAM_WEIGHT_DECAY = 3e-5
GRAD_CLIP = 0.95
TRAIN_RULE = True
TRAIN_PROGRAM = True
PERCEPTION_NOISE_START = 0.0
PERCEPTION_NOISE_END = 0.0
SEED = 0
DEVICE = "auto"
VALIDATION_EVERY = 200
CHECKPOINT_EVERY = 200

# Explicit deterministic post-training cases
TEST_CASES = (
    TestCase("train_large", 11, 8, 66, 99),
    TestCase("longer_tape", 14, 8, 84, 126),
    TestCase("longer_tape_input", 14, 11, 84, 126),
)
EVALUATION_BATCH_SIZE = 256
EVALUATION_MAX_EXAMPLES = None

geometry = GeometryConfig(
    stride=STRIDE,
    vertical_space=VERTICAL_SPACE,
    horizontal_space=HORIZONTAL_SPACE,
    program_start=PROGRAM_START,
    wrap_y=WRAP_Y,
)
model_config = ModelConfig(
    program_channels=PROGRAM_CHANNELS,
    computation_channels=COMPUTATION_CHANNELS,
    program_mode=PROGRAM_MODE,
    program_init_std=PROGRAM_INIT_STD,
    hidden_size=HIDDEN_SIZE,
    fixed_kernels=FIXED_KERNELS,
    fixed_laplacian=FIXED_LAPLACIAN,
    learnable_kernels=LEARNABLE_KERNELS,
    learnable_kernel_init=LEARNABLE_KERNEL_INIT,
    activation=ACTIVATION,
    gate=GATE,
    gate_bias=GATE_BIAS,
    fire_rate=FIRE_RATE,
    state_leak=STATE_LEAK,
    max_abs_state=MAX_ABS_STATE,
    random_kernel_seed=SEED,
)
training_config = TrainingConfig(
    updates=UPDATES,
    batch_size_per_task=BATCH_SIZE_PER_TASK,
    base_tape_slots=BASE_TAPE_SLOTS,
    base_input_max_lengths=BASE_INPUT_MAX_LENGTHS,
    tape_variation=TAPE_VARIATION,
    input_variation=INPUT_VARIATION,
    free_steps_per_tape_slot=FREE_STEPS_PER_TAPE_SLOT,
    time_variation=TIME_VARIATION,
    supervision_ratio=SUPERVISION_RATIO,
    lr_points=LR_POINTS,
    lr_interpolation=LR_INTERPOLATION,
    weight_decay=WEIGHT_DECAY,
    program_weight_decay=PROGRAM_WEIGHT_DECAY,
    grad_clip=GRAD_CLIP,
    train_rule=TRAIN_RULE,
    train_program=TRAIN_PROGRAM,
    perception_noise_start=PERCEPTION_NOISE_START,
    perception_noise_end=PERCEPTION_NOISE_END,
    seed=SEED,
    validation_every=VALIDATION_EVERY,
    checkpoint_every=CHECKPOINT_EVERY,
    device=DEVICE,
)
config = ExperimentConfig(geometry, model_config, training_config, TEST_CASES)
checkpoint_dir = PROJECT_ROOT / "checkpoints" / (
    f"{EXPERIMENT_NAME}_{PROGRAM_MODE}_trials_{len(BASE_TAPE_SLOTS)}"
)
checkpoint_path = checkpoint_dir / "best.pt"

report = validate_experiment(config, TASK_NAMES)
print(report)
print(
    f"channels: program=0:{PROGRAM_CHANNELS}, shared I/O={model_config.io_channel}, "
    f"computation={model_config.io_channel + 1}:{model_config.channels}, "
    f"total={model_config.channels}"
)
print(
    f"program tile per task: ({PROGRAM_CHANNELS}, {geometry.height}, {STRIDE}); "
    f"mode={PROGRAM_MODE}; origin x={PROGRAM_START}"
)
print(f"task weights: {dict(TASKS)}")
print(f"learning-rate schedule: {LR_INTERPOLATION} {LR_POINTS}")
for trial in sorted(report.sample_trials, key=lambda item: item.tape_slots):
    layout = TapeLayout(geometry, trial.tape_slots)
    left = layout.tape_start
    right = layout.width - 1 - layout.tape_coordinates[-1][1]
    print(
        f"\nSampled trial: tape={trial.tape_slots}, input<=L{trial.input_max_length}, "
        f"steps={trial.free_steps}+{trial.supervision_steps}; margins={left}/{right}\n"
        f"{layout.schema()}"
    )
preview_tasks = binary_tasks(TASK_NAMES, min(2, max(BASE_INPUT_MAX_LENGTHS)))
preview = MultiTaskDataset.from_tasks(preview_tasks, max(3, min(BASE_TAPE_SLOTS)))
print("\nOrdered examples (empty first; B is supervised blank padding):")
for index, source in enumerate(preview.datasets[0].input_strings):
    padded = (source + "B" * preview.tape_slots)[:preview.tape_slots]
    targets = {
        dataset.name: (dataset.target_strings[index] + "B" * preview.tape_slots)[:preview.tape_slots]
        for dataset in preview.datasets
    }
    print(padded, targets)


In [ ]:
RUN_TRAINING = True
LOAD_CHECKPOINT = False
RESUME_LATEST = False
PROGRESS_EVERY = 25
LOSS_RUNNING_MEAN = 100


def training_plots(trainer):
    loss_window = deque(maxlen=LOSS_RUNNING_MEAN)
    training_updates, running_losses, learning_rates = [], [], []
    validation_updates = []
    validation_accuracies = {name: [] for name in TASK_NAMES}

    def record(update, loss, learning_rate, accuracies):
        loss_window.append(float(loss))
        training_updates.append(int(update))
        running_losses.append(max(sum(loss_window) / len(loss_window), 1e-12))
        learning_rates.append(float(learning_rate))
        if accuracies is not None:
            validation_updates.append(int(update))
            for name in TASK_NAMES:
                validation_accuracies[name].append(float(accuracies[name]))

    for entry in trainer.history:
        record(
            entry["update"], entry["loss"], entry["learning_rate"],
            entry.get("validation_accuracies"),
        )

    validation_figure, validation_axis = plt.subplots(figsize=(9, 4.5))
    validation_lines = {
        name: validation_axis.plot([], [], label=name)[0] for name in TASK_NAMES
    }
    validation_axis.set(
        title="Periodic validation: exact semantic accuracy",
        xlabel="optimizer update",
        ylabel="accuracy",
        xlim=(0, UPDATES),
        ylim=(0, 1.01),
    )
    validation_axis.yaxis.set_major_formatter(PercentFormatter(1.0))
    validation_axis.grid(alpha=0.25)
    validation_axis.legend(ncol=2)

    training_figure, loss_axis = plt.subplots(figsize=(9, 4.5))
    learning_rate_axis = loss_axis.twinx()
    loss_line = loss_axis.plot([], [], color="tab:blue", label="running loss")[0]
    learning_rate_line = learning_rate_axis.plot(
        [], [], color="tab:orange", label="learning rate"
    )[0]
    loss_axis.set(
        title=f"Training loss ({LOSS_RUNNING_MEAN}-update running mean) and learning rate",
        xlabel="optimizer update",
        ylabel="weighted MSE",
        xlim=(0, UPDATES),
    )
    learning_rate_axis.set_ylabel("learning rate")
    loss_axis.set_yscale("log")
    learning_rate_axis.set_yscale("log")
    loss_axis.set_ylim(1e-6, 1.0)
    schedule_rates = [rate for _, rate in LR_POINTS]
    learning_rate_axis.set_ylim(min(schedule_rates) / 2, max(schedule_rates) * 2)
    loss_axis.grid(alpha=0.25)
    loss_axis.legend(
        [loss_line, learning_rate_line],
        [loss_line.get_label(), learning_rate_line.get_label()],
    )

    validation_display = display(validation_figure, display_id=True)
    training_display = display(training_figure, display_id=True)
    plt.close(validation_figure)
    plt.close(training_figure)

    def update_plots(metrics):
        record(
            metrics.update, metrics.loss, metrics.learning_rate,
            metrics.validation_accuracies,
        )
        if metrics.validation_accuracies is not None:
            for name, line in validation_lines.items():
                line.set_data(validation_updates, validation_accuracies[name])
            validation_display.update(validation_figure)
        if metrics.update % PROGRESS_EVERY == 0 or metrics.validation_loss is not None:
            loss_line.set_data(training_updates, running_losses)
            learning_rate_line.set_data(training_updates, learning_rates)
            loss_axis.set_autoscaley_on(True)
            learning_rate_axis.set_autoscaley_on(True)
            loss_axis.relim()
            loss_axis.autoscale_view(scalex=False)
            learning_rate_axis.relim()
            learning_rate_axis.autoscale_view(scalex=False)
            training_display.update(training_figure)

    return update_plots


if RUN_TRAINING:
    latest = checkpoint_dir / "latest.pt"
    if RESUME_LATEST:
        if not latest.is_file():
            raise FileNotFoundError(latest)
        trainer = Trainer.from_checkpoint(latest, TASKS, device=DEVICE)
        if trainer.config != config:
            raise ValueError("checkpoint configuration differs from this notebook")
    else:
        trainer = Trainer(config, TASKS)
    plot_callback = training_plots(trainer)
    trainer.fit(
        checkpoint_dir, progress_every=PROGRESS_EVERY, callback=plot_callback
    )
    model = trainer.model
elif LOAD_CHECKPOINT:
    if not checkpoint_path.is_file():
        raise FileNotFoundError(checkpoint_path)
    model, loaded_config, checkpoint = load_model(checkpoint_path, DEVICE)
    stored_specs = tuple(tuple(spec) for spec in checkpoint["task_specs"])
    if loaded_config != config or stored_specs != TASKS:
        raise ValueError("checkpoint configuration or task weights differ")
else:
    device = torch.device(
        "cuda" if DEVICE == "auto" and torch.cuda.is_available()
        else "cpu" if DEVICE == "auto" else DEVICE
    )
    torch.manual_seed(SEED)
    model = NeuralCellularAutomaton(model_config, geometry, TASK_NAMES).to(device)

print(f"model device: {model.device}; parameters: {model.parameter_count:,}")


In [ ]:
RUN_EVALUATION = True

if RUN_EVALUATION:
    evaluation_results = evaluate_cases(
        model,
        geometry,
        TASK_NAMES,
        TEST_CASES,
        batch_size=EVALUATION_BATCH_SIZE,
        seed=SEED,
        max_examples=EVALUATION_MAX_EXAMPLES,
    )
    print(format_results(evaluation_results))


In [ ]:
RUN_VISUALIZATION = True
VISUALIZATION_TASK = "reverse"
VISUALIZATION_INPUT = "1100"
VISUALIZATION_TAPE_SLOTS = 11
VISUALIZATION_FREE_STEPS = 80
VISUALIZATION_SUPERVISION_STEPS = int(VISUALIZATION_FREE_STEPS * SUPERVISION_RATIO)
GIF_DURATION_MS = 80
GIF_SCALE = 36
GIF_PATH = PROJECT_ROOT / "outputs" / "output.gif"

if RUN_VISUALIZATION:
    if VISUALIZATION_TASK not in TASK_NAMES:
        raise ValueError("VISUALIZATION_TASK is not trained")
    if len(VISUALIZATION_INPUT) > VISUALIZATION_TAPE_SLOTS - 1:
        raise ValueError("visualized input must leave at least one blank tape slot")
    task = binary_tasks(
        (VISUALIZATION_TASK,), len(VISUALIZATION_INPUT), include_shorter=False
    )[0]
    target_symbols = next(
        example.target for example in task.examples
        if example.input == VISUALIZATION_INPUT
    )
    steps = VISUALIZATION_FREE_STEPS + VISUALIZATION_SUPERVISION_STEPS
    layout = TapeLayout(geometry, VISUALIZATION_TAPE_SLOTS)
    encoded = encode_strings((VISUALIZATION_INPUT,), VISUALIZATION_TAPE_SLOTS).to(model.device)
    task_indices = torch.tensor([model.task_index(VISUALIZATION_TASK)], device=model.device)
    with torch.no_grad():
        trajectory = model(
            model.initial_state(layout.render_tape(encoded), task_indices), steps
        )[0]
    print(interpret_tape(
        layout.extract_tape(trajectory[-1, model.config.io_channel]).cpu(),
        task.output_mode,
    ))
    gif_path = save_gif(
        trajectory,
        GIF_PATH,
        layout=layout,
        config=config,
        program_tile=model.programs[model.task_index(VISUALIZATION_TASK)],
        task_name=VISUALIZATION_TASK,
        input_symbols=VISUALIZATION_INPUT,
        target_symbols=target_symbols,
        free_steps=VISUALIZATION_FREE_STEPS,
        supervision_steps=VISUALIZATION_SUPERVISION_STEPS,
        output_mode=task.output_mode,
        duration_ms=GIF_DURATION_MS,
        scale=GIF_SCALE,
    )
    display(DisplayImage(filename=str(gif_path)))


In [ ]:
# Learned program tiles, shown with one shared color scale
import matplotlib.pyplot as plt

program_tiles = model.programs.detach().float().cpu()
task_count, channel_count, tile_height, tile_width = program_tiles.shape
color_limit = max(float(program_tiles.abs().max()), 1e-6)
figure, axes = plt.subplots(
    task_count,
    channel_count,
    figsize=(2.4 * channel_count + 1.0, 1.55 * task_count),
    squeeze=False,
    constrained_layout=True,
)

for task_index, task_name in enumerate(model.task_names):
    for channel in range(channel_count):
        axis = axes[task_index, channel]
        tile = program_tiles[task_index, channel]
        image = axis.imshow(
            tile, cmap="coolwarm", vmin=-color_limit, vmax=color_limit,
            interpolation="nearest", aspect="equal",
        )
        axis.set_xticks(range(tile_width))
        axis.set_yticks(range(tile_height))
        if task_index == 0:
            axis.set_title(f"program channel {channel}")
        if channel == 0:
            axis.set_ylabel(task_name, rotation=0, ha="right", va="center")

figure.colorbar(
    image, ax=axes.ravel().tolist(), shrink=0.75, pad=0.02, label="program value"
)
figure.suptitle("Learned repeated program tile for each task")
plt.show()
